# Notebook 02 — Generate the German Question Set with Ollama (full-document version)

## Purpose

This notebook turns the six German Botschaften prepared by Notebook 01 into a question set for the thesis. In this version the model reads the **complete Botschaft in one prompt** (no chunking), so questions can draw on any part of the document, including comparative questions that connect different sections.

Expected project layout:

```text
project/
├── data/
│   ├── extracted_text/          # from Notebook 01
│   └── questions/               # written by this notebook
│       └── cache/               # raw model responses, lets you resume an interrupted run
└── notebooks/
```

Run from the project root, or change `PROJECT_ROOT` below.

## 1. Configuration

All run settings live in this cell and are saved to `data/questions/run_config.json` at the end, so the exact configuration of the run can be reported in the thesis.

Notes on the choices:

* **`num_ctx` is not set here.** It is computed in Section 3 from the longest document, then used for *every* call. Changing `num_ctx` between calls would make Ollama reload the model.
* **`THINK = False`** asks Ollama to disable the reasoning phase for these extraction-style steps. If the installed `qwen3-next:80b` build is a thinking-only variant, the notebook detects the refusal, continues without the flag, and strips any `<think>…</think>` block from the response. In that case raise `MAX_OUTPUT_TOKENS_GENERATION` (reasoning tokens count against the output budget).
* **Temperatures** are low because the tasks are extraction and selection. The model's own defaults (temperature 0.6, top_k 20, top_p 0.95) are left unchanged for the other sampling parameters and are recorded in the run configuration.
* **`QUOTA`** fixes the question-type mix per Botschaft (4 factual, 3 explanatory, 3 comparative = 10, inside the planned 8–10 range).

In [14]:
from pathlib import Path
import json
import math
import re
import time
import unicodedata
import requests
import pandas as pd

# ------------------------------------------------------------
# Model and server
# ------------------------------------------------------------
MODEL_NAME = "qwen3-next:80b"          # change this variable to switch models
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
KEEP_ALIVE = "60m"                     # keep the 50 GB model loaded between calls

# ------------------------------------------------------------
# Decoding settings (recorded in run_config.json)
# ------------------------------------------------------------
SEED = 42
TEMPERATURE_GENERATION = 0.2
TEMPERATURE_SELECTION = 0.0
TEMPERATURE_TRANSLATION = 0.0
THINK = False                          # False = no reasoning phase; None = leave to the model default
MAX_OUTPUT_TOKENS_GENERATION = 8000    # includes reasoning tokens if the model cannot disable thinking
MAX_OUTPUT_TOKENS_SELECTION = 2000
MAX_OUTPUT_TOKENS_TRANSLATION = 600

# ------------------------------------------------------------
# Context window
# ------------------------------------------------------------
NUM_CTX_OVERRIDE = None                # set an int to force a value, otherwise computed in Section 3
TOKEN_CHARS_CONSERVATIVE = 3.0         # characters per token used for sizing; German text is usually ~3.5-4
PROMPT_OVERHEAD_TOKENS = 3000          # instructions + JSON schema

# ------------------------------------------------------------
# Question design
# ------------------------------------------------------------
QUESTION_TYPES = ["factual", "explanatory", "comparative"]
CANDIDATES_PER_TYPE = 6                # candidates requested per type and document
QUOTA = {"factual": 4, "explanatory": 3, "comparative": 3}   # selected per document
MIN_QUESTIONS_PER_DOCUMENT = 8
MAX_QUESTIONS_PER_DOCUMENT = 10
ALLOW_PARTIAL_EVIDENCE = True          # keep candidates whose quotation is only partly found (e.g. spans a page break)
DUPLICATE_JACCARD_THRESHOLD = 0.8

# ------------------------------------------------------------
# Robustness and bookkeeping
# ------------------------------------------------------------
REQUEST_TIMEOUT_SECONDS = 3600         # a full-document prompt can take minutes to prefill
MAX_RETRIES = 3
USE_CACHE = True                       # reuse saved raw responses if the same call was already made
PROMPT_VERSION = "v2-fulldoc-2026-10"

PROJECT_ROOT = Path.cwd()
TEXT_DIR = PROJECT_ROOT / "../data" / "extracted_text"
QUESTION_DIR = PROJECT_ROOT / "../data" / "questions"
CACHE_DIR = QUESTION_DIR / "cache"
for d in (QUESTION_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert sum(QUOTA.values()) <= MAX_QUESTIONS_PER_DOCUMENT
assert sum(QUOTA.values()) >= MIN_QUESTIONS_PER_DOCUMENT
assert set(QUOTA) == set(QUESTION_TYPES)

print("Project root:", PROJECT_ROOT)
print("Model:", MODEL_NAME, "| prompt version:", PROMPT_VERSION)
print("Quota per document:", QUOTA, "=", sum(QUOTA.values()))

Project root: /home/bern/Documents/llm_trustworthiness/scripts
Model: qwen3-next:80b | prompt version: v2-fulldoc-2026-10
Quota per document: {'factual': 4, 'explanatory': 3, 'comparative': 3} = 10


## 2. Check Ollama, the model and the corpus

This cell verifies that Ollama is reachable, the model is installed, and reads the model's **digest, quantisation, capabilities and maximum context length** from Ollama itself, so that the numbers in the thesis come from the server and not from memory.

In [15]:
def ollama_get(path):
    r = requests.get(f"{OLLAMA_BASE_URL}{path}", timeout=60)
    r.raise_for_status()
    return r.json()

def ollama_post(path, payload):
    r = requests.post(f"{OLLAMA_BASE_URL}{path}", json=payload, timeout=120)
    r.raise_for_status()
    return r.json()

tags = ollama_get("/api/tags").get("models", [])
installed = {m["name"]: m for m in tags}
print("Ollama is reachable. Installed models:", len(installed))

if MODEL_NAME not in installed:
    raise RuntimeError(
        f"Model {MODEL_NAME!r} is not installed. Installed: {sorted(installed)}. "
        "Run `ollama list` in the server terminal and update MODEL_NAME."
    )

MODEL_DIGEST = installed[MODEL_NAME].get("digest", "")
show = ollama_post("/api/show", {"model": MODEL_NAME})
details = show.get("details", {})
model_info = show.get("model_info", {})
ctx_keys = [k for k in model_info if k.endswith(".context_length")]
MODEL_MAX_CTX = int(model_info[ctx_keys[0]]) if ctx_keys else 262144
try:
    OLLAMA_VERSION = ollama_get("/api/version").get("version", "unknown")
except Exception:
    OLLAMA_VERSION = "unknown"

print("Model digest      :", MODEL_DIGEST[:12])
print("Parameter size    :", details.get("parameter_size"))
print("Quantisation      :", details.get("quantization_level"))
print("Capabilities      :", show.get("capabilities"))
print("Max context length:", MODEL_MAX_CTX)
print("Model defaults    :", " | ".join(show.get("parameters", "").split("\n")))
print("Ollama version    :", OLLAMA_VERSION)

if not TEXT_DIR.exists():
    raise FileNotFoundError(f"Could not find {TEXT_DIR}. Run Notebook 01 first from the same project root.")
json_files = sorted(TEXT_DIR.glob("*.json"))
txt_files = sorted(TEXT_DIR.glob("*.txt"))
if not json_files and not txt_files:
    raise FileNotFoundError(f"No .txt or .json corpus files found in {TEXT_DIR}.")
print(f"Found {len(txt_files)} TXT files and {len(json_files)} JSON files.")

Ollama is reachable. Installed models: 19
Model digest      : b2ebb986e4e9
Parameter size    : 79.7B
Quantisation      : Q4_K_M
Capabilities      : ['completion', 'tools', 'thinking']
Max context length: 262144
Model defaults    : top_k                          20 | top_p                          0.95 | repeat_penalty                 1 | stop                           "<|im_start|>" | stop                           "<|im_end|>" | temperature                    0.6
Ollama version    : 0.24.0
Found 6 TXT files and 6 JSON files.


## 3. Load the Botschaften and size the context window

The JSON files from Notebook 01 are preferred because they keep page numbers. Every document is assembled into one text with `[PAGE n]` markers.

Then `num_ctx` is computed once from the **longest** document (conservatively estimated at 3 characters per token) plus prompt overhead and the output budget, rounded up to a multiple of 4096 and capped at the model maximum. The same value is used for all calls. If the longest document does not fit, the notebook stops instead of letting Ollama silently truncate the prompt.

In [16]:
def load_json_document(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    pages = []
    for page in payload.get("pages", []):
        text = (page.get("text") or "").strip()
        if text:
            pages.append({"page": int(page["page"]) if page.get("page") is not None else None, "text": text})
    return {
        "document_id": payload.get("selection_key") or path.stem,
        "title": payload.get("title") or path.stem,
        "date": payload.get("date"),
        "source_file": path.name,
        "pages": pages,
    }

def parse_txt_pages(path: Path) -> dict:
    text = path.read_text(encoding="utf-8", errors="replace")
    title_match = re.search(r"^TITLE:\s*(.+)$", text, flags=re.MULTILINE)
    matches = list(re.finditer(r"===== PAGE (\d+) =====", text))
    pages = []
    for i, m in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        page_text = text[m.end():end].strip()
        if page_text:
            pages.append({"page": int(m.group(1)), "text": page_text})
    if not pages:
        pages = [{"page": None, "text": text}]
    return {
        "document_id": path.stem,
        "title": title_match.group(1).strip() if title_match else path.stem,
        "date": None,
        "source_file": path.name,
        "pages": pages,
    }

def assemble_text(pages):
    return "\n\n".join(
        (f"[PAGE {p['page']}]\n" if p["page"] is not None else "") + p["text"] for p in pages
    )

documents, used = [], set()
for jp in json_files:
    try:
        doc = load_json_document(jp)
        if doc["pages"]:
            documents.append(doc)
            used.add(jp.stem)
    except (json.JSONDecodeError, OSError) as exc:
        print(f"WARNING: could not read {jp.name}: {exc}")
for tp in txt_files:
    if tp.stem not in used:
        doc = parse_txt_pages(tp)
        if doc["pages"]:
            documents.append(doc)
if not documents:
    raise RuntimeError("No readable document text could be loaded.")

documents = sorted(documents, key=lambda d: (d["title"].casefold(), d["document_id"]))
for doc in documents:
    doc["text"] = assemble_text(doc["pages"])
    doc["est_tokens"] = int(len(doc["text"]) / TOKEN_CHARS_CONSERVATIVE)

longest = max(d["est_tokens"] for d in documents)
needed = longest + PROMPT_OVERHEAD_TOKENS + MAX_OUTPUT_TOKENS_GENERATION
if NUM_CTX_OVERRIDE:
    NUM_CTX = int(NUM_CTX_OVERRIDE)
else:
    NUM_CTX = min(MODEL_MAX_CTX, int(math.ceil(needed / 4096) * 4096))
if needed > min(NUM_CTX, MODEL_MAX_CTX):
    raise RuntimeError(
        f"The longest document needs about {needed:,} tokens (incl. overhead and output) but num_ctx is "
        f"{min(NUM_CTX, MODEL_MAX_CTX):,}. Reduce MAX_OUTPUT_TOKENS_GENERATION or fall back to chunking."
    )

overview = pd.DataFrame([{
    "document_id": d["document_id"], "title": d["title"], "pages_with_text": len(d["pages"]),
    "characters": len(d["text"]), "est_tokens_conservative": d["est_tokens"],
    "share_of_num_ctx": round(d["est_tokens"] / NUM_CTX, 2),
} for d in documents])
display(overview)
print(f"num_ctx for ALL calls: {NUM_CTX:,}  (longest document ≈ {longest:,} tokens, model maximum {MODEL_MAX_CTX:,})")

,document_id,title,pages_with_text,characters,est_tokens_conservative,share_of_num_ctx
0,13_ahv_rente,Botschaft zur Volksinitiative «Für ein bessere...,28,74703,24901,0.22
1,umweltverantwortungsinitiative,Botschaft zur Volksinitiative «Für eine verant...,20,50122,16707,0.15
2,steuergerechtigkeits_initiative,Botschaft zur Volksinitiative «Für eine zivils...,108,274951,91650,0.80
3,stopfleber_initiative,Botschaft zur Volksinitiative «Ja zum Importve...,16,44034,14678,0.13
4,nachhaltigkeitsinitiative,Botschaft zur Volksinitiative «Keine 10-Millio...,110,299015,99671,0.87
5,neutralitaetsinitiative,Botschaft zur Volksinitiative «Wahrung der sch...,30,80586,26862,0.23


num_ctx for ALL calls: 114,688  (longest document ≈ 99,671 tokens, model maximum 262,144)


## 4. Helper functions: Ollama calls, JSON parsing, evidence verification

* `call_ollama` sends every request with the same `num_ctx`, seed and keep-alive, handles models that refuse the `think` flag, strips `<think>` blocks, retries on malformed JSON, and logs token counts and durations to `CALL_LOG`.
* Raw responses are cached on disk. If a long run is interrupted, re-running the notebook reuses finished calls (switch off with `USE_CACHE = False`, or change `PROMPT_VERSION`, to force fresh calls).
* Evidence verification normalises both the quotation and the page text to lowercase letters and digits only, so line-break hyphenation, spacing, punctuation and quotation-mark style do not cause false mismatches.

In [17]:
CALL_LOG = []
_THINK_SUPPORTED = {"flag": THINK is not None}

def strip_think(text: str) -> str:
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def extract_json_object(text: str):
    text = strip_think(text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise

def safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value)[:150]

def call_ollama(prompt, *, temperature, num_predict, label):
    est_prompt_tokens = len(prompt) / TOKEN_CHARS_CONSERVATIVE
    if est_prompt_tokens + num_predict > NUM_CTX:
        raise RuntimeError(
            f"[{label}] prompt ≈{int(est_prompt_tokens):,} tokens + output {num_predict:,} exceeds num_ctx {NUM_CTX:,}."
        )

    cache_file = CACHE_DIR / f"{safe_name(MODEL_NAME)}__{PROMPT_VERSION}__{safe_name(label)}.json"
    if USE_CACHE and cache_file.exists():
        return json.loads(cache_file.read_text(encoding="utf-8"))

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "keep_alive": KEEP_ALIVE,
        "options": {
            "temperature": temperature,
            "num_ctx": NUM_CTX,
            "num_predict": num_predict,
            "seed": SEED,
        },
    }
    last_error = None
    attempt = 0
    while attempt < MAX_RETRIES:
        body = dict(payload)
        if _THINK_SUPPORTED["flag"] and THINK is not None:
            body["think"] = THINK
        try:
            started = time.time()
            r = requests.post(OLLAMA_GENERATE_URL, json=body, timeout=(30, REQUEST_TIMEOUT_SECONDS))
            if r.status_code == 400 and "think" in r.text.lower() and "think" in body:
                print(f"[{label}] this model build does not accept the think flag; continuing without it.")
                _THINK_SUPPORTED["flag"] = False
                continue                      # retry immediately without counting an attempt
            r.raise_for_status()
            result = r.json()
            CALL_LOG.append({
                "label": label,
                "prompt_eval_count": result.get("prompt_eval_count"),
                "eval_count": result.get("eval_count"),
                "done_reason": result.get("done_reason"),
                "total_seconds": round(result.get("total_duration", 0) / 1e9, 1),
                "wall_seconds": round(time.time() - started, 1),
            })
            if result.get("done_reason") == "length":
                raise RuntimeError("output hit the token limit (raise the MAX_OUTPUT_TOKENS_* value)")
            raw = (result.get("response") or "")
            if not strip_think(raw):
                raise RuntimeError("empty response (the model may have used the whole budget for reasoning)")
            parsed = extract_json_object(raw)
            cache_file.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
            return parsed
        except Exception as exc:
            attempt += 1
            last_error = exc
            print(f"[{label}] request failed (attempt {attempt}/{MAX_RETRIES}): {exc}")
            if attempt < MAX_RETRIES:
                time.sleep(3 * attempt)
    raise RuntimeError(f"[{label}] failed after {MAX_RETRIES} attempts: {last_error}")


# ---------- evidence verification ----------
def alnum_norm(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"[\W_]+", "", text)

def build_page_index(doc):
    return {p["page"]: alnum_norm(p["text"]) for p in doc["pages"] if p["page"] is not None}

def verify_quote(quote, cited_page, page_index):
    # Returns (status, pages_found); status: exact | exact_other_page | partial | not_found
    q = alnum_norm(quote)
    if len(q) < 25:
        return "not_found", []
    found = [pg for pg, txt in page_index.items() if q in txt]
    if found:
        try:
            cited = int(cited_page)
        except (TypeError, ValueError):
            cited = None
        return ("exact" if cited in found else "exact_other_page"), found
    # partial: start and end of the quotation both occur (e.g. quotation spans a page break)
    head, tail = q[:40], q[-40:]
    pages_head = [pg for pg, txt in page_index.items() if head in txt]
    pages_tail = [pg for pg, txt in page_index.items() if tail in txt]
    if pages_head and pages_tail:
        return "partial", sorted(set(pages_head + pages_tail))
    return "not_found", []

def word_set(text):
    return set(re.findall(r"\w+", text.casefold()))

def jaccard(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0

## 5. Generate German candidate questions from the complete Botschaft

For each Botschaft there is **one call per question type** (factual, explanatory, comparative). The Botschaft comes first in the prompt and the instructions last, so the three calls share an identical prefix and Ollama can reuse the already-processed document instead of reading it three times.

The model does **not** write a reference answer. For every candidate it returns only:

* the question (German),
* one to three **verbatim evidence quotations** with their page numbers.

Comparative questions may cite quotations from different pages, which is the main benefit of giving the model the whole document.

In [18]:
TYPE_GUIDE = {
    "factual": (
        "factual recall: a specific fact, actor, date, figure, measure or decision that is stated in the Botschaft "
        "(who/what/when/how many/which measure)."
    ),
    "explanatory": (
        "explanatory: the reasons, aims, mechanisms or consequences that the Botschaft itself gives for something "
        "(why/how/for what purpose/with what effect), attributed to whoever states them."
    ),
    "comparative": (
        "comparative or relational: a comparison or relationship that the Botschaft explicitly draws or clearly supports, "
        "for example the position of the initiative committee versus the position of the Federal Council, the current "
        "situation versus the proposed change, or two measures/options against each other. Use evidence from at least "
        "two different places in the document where possible."
    ),
}

def document_block(doc):
    return (
        f"DOKUMENTTITEL: {doc['title']}\n"
        f"DATUM: {doc.get('date') or 'unbekannt'}\n\n"
        f"<BOTSCHAFT>\n{doc['text']}\n</BOTSCHAFT>"
    )

def candidate_instructions(question_type, n):
    return f'''You are helping to construct a research question set for evaluating answers that are grounded in official Swiss Federal Council Botschaften. The complete Botschaft is given above. Page numbers are marked as [PAGE n].

Use ONLY the Botschaft above. Do not use outside knowledge and do not invent facts.

Write {n} distinct candidate questions of this type: {TYPE_GUIDE[question_type]}

Requirements:
- Write every question in German.
- Each question must be clear, neutral and answerable from the Botschaft alone.
- Each question must be understandable without the document at hand: name the initiative or measure explicitly and never write phrases such as "im Text", "in diesem Abschnitt" or "gemäss der Botschaft".
- Spread the questions over different parts of the document; do not cluster them on one page or topic.
- Avoid yes/no questions, vague questions, near-duplicates and questions that need outside knowledge.
- Do not present political positions as facts. If a position belongs to the Federal Council, the initiative committee, Parliament or another actor, name that actor in the question.
- Do NOT write answers or reference answers.
- For each question give 1 to 3 evidence quotations. Each quotation must be copied verbatim from a single page of the Botschaft (8 to 40 words, exactly as printed, without ellipses) and must contain the information needed to answer the question. Give the page number from the nearest [PAGE n] marker above the quotation.

Return valid JSON only, in exactly this structure:
{{"questions":[{{"question_de":"...","evidence":[{{"quote_de":"...","page":12}}]}}]}}'''

candidate_rows = []
for d_idx, doc in enumerate(documents, start=1):
    page_index = build_page_index(doc)
    prefix = document_block(doc)
    print(f"[{d_idx}/{len(documents)}] {doc['title']}  (≈{doc['est_tokens']:,} tokens)")
    for qtype in QUESTION_TYPES:
        prompt = prefix + "\n\n" + candidate_instructions(qtype, CANDIDATES_PER_TYPE)
        result = call_ollama(
            prompt, temperature=TEMPERATURE_GENERATION, num_predict=MAX_OUTPUT_TOKENS_GENERATION,
            label=f"{doc['document_id']}__candidates__{qtype}",
        )
        items = result.get("questions", [])
        if not isinstance(items, list):
            print(f"   WARNING: unexpected output for {qtype}")
            continue
        n_ok = 0
        for item in items:
            q = str(item.get("question_de", "")).strip()
            evidence = item.get("evidence", [])
            if not q or not isinstance(evidence, list) or not evidence:
                continue
            checks, quotes, pages = [], [], []
            for ev in evidence[:3]:
                if not isinstance(ev, dict):
                    continue
                quote = str(ev.get("quote_de", "")).strip()
                status, found_pages = verify_quote(quote, ev.get("page"), page_index)
                checks.append(status)
                quotes.append(quote)
                pages.extend(found_pages if found_pages else ([int(ev["page"])] if str(ev.get("page", "")).isdigit() else []))
            if all(c in ("exact", "exact_other_page") for c in checks):
                evidence_status = "verified"
            elif all(c in ("exact", "exact_other_page", "partial") for c in checks):
                evidence_status = "partial"
            else:
                evidence_status = "failed"
            n_ok += evidence_status == "verified"
            candidate_rows.append({
                "document_id": doc["document_id"],
                "document_title": doc["title"],
                "source_file": doc["source_file"],
                "question_type": qtype,
                "question_de": q,
                "evidence_quotes_de": json.dumps(quotes, ensure_ascii=False),
                "source_pages": json.dumps(sorted(set(pages))),
                "evidence_status": evidence_status,
                "evidence_checks": json.dumps(checks),
            })
        print(f"   {qtype:12s}: {len(items)} proposed, {n_ok} with verified evidence")

candidates = pd.DataFrame(candidate_rows)
if candidates.empty:
    raise RuntimeError("No candidate questions were generated. Inspect the cache folder and CALL_LOG.")

# duplicate removal within a document (exact normalised text and word-overlap)
keep = []
for doc_id, grp in candidates.groupby("document_id", sort=False):
    kept_sets = []
    for idx, row in grp.iterrows():
        ws = word_set(row["question_de"])
        if any(jaccard(ws, other) >= DUPLICATE_JACCARD_THRESHOLD for other in kept_sets):
            continue
        kept_sets.append(ws)
        keep.append(idx)
candidates = candidates.loc[keep].reset_index(drop=True)
candidates.insert(0, "candidate_id", range(len(candidates)))

candidate_path = QUESTION_DIR / "question_candidates_german.csv"
candidates.to_csv(candidate_path, index=False, encoding="utf-8-sig")
print(f"\nSaved {len(candidates)} unique candidates (all evidence statuses) to: {candidate_path}")
display(candidates.groupby(["document_title", "question_type", "evidence_status"]).size().rename("n").reset_index())

[1/6] Botschaft zur Volksinitiative «Für ein besseres Leben im Alter (Initiative für eine 13. AHV-Rente)»  (≈24,901 tokens)
   factual     : 6 proposed, 6 with verified evidence
   explanatory : 6 proposed, 5 with verified evidence
   comparative : 6 proposed, 6 with verified evidence
[2/6] Botschaft zur Volksinitiative «Für eine verantwortungsvolle Wirtschaft innerhalb der planetaren Grenzen (Umweltverantwortungsinitiative)»  (≈16,707 tokens)
   factual     : 6 proposed, 5 with verified evidence
   explanatory : 6 proposed, 4 with verified evidence
   comparative : 6 proposed, 2 with verified evidence
[3/6] Botschaft zur Volksinitiative «Für eine zivilstandsunabhängige Individualbesteuerung (Steuergerechtigkeits-Initiative)» und zum indirekten Gegenvorschlag (Bundesgesetz über die Individualbesteuerung)  (≈91,650 tokens)
   factual     : 6 proposed, 6 with verified evidence
   explanatory : 6 proposed, 5 with verified evidence
   comparative : 6 proposed, 6 with verified evidence
[4/6

,document_title,question_type,evidence_status,n
0,Botschaft zur Volksinitiative «Für ein bessere...,comparative,verified,6
1,Botschaft zur Volksinitiative «Für ein bessere...,explanatory,partial,1
2,Botschaft zur Volksinitiative «Für ein bessere...,explanatory,verified,5
3,Botschaft zur Volksinitiative «Für ein bessere...,factual,verified,6
4,Botschaft zur Volksinitiative «Für eine verant...,comparative,failed,1
5,Botschaft zur Volksinitiative «Für eine verant...,comparative,partial,2
6,Botschaft zur Volksinitiative «Für eine verant...,comparative,verified,2
7,Botschaft zur Volksinitiative «Für eine verant...,explanatory,failed,2
8,Botschaft zur Volksinitiative «Für eine verant...,explanatory,verified,3
9,Botschaft zur Volksinitiative «Für eine verant...,factual,partial,1


## 6. Select the question set per Botschaft

Only candidates whose quotation was found in the source are eligible (`verified`, plus `partial` if `ALLOW_PARTIAL_EVIDENCE` is on). The model picks the best candidates **per question type** and the number per type is fixed by `QUOTA`, so the type mix is enforced in code instead of only requested in a prompt.

The selection prompt does not contain the document: it only sees the candidate questions and their page numbers, and it may not rewrite anything. Invalid or missing IDs fall back to the first eligible candidates of that type. Shortfalls (fewer eligible candidates than the quota) are reported, never filled with invented questions.

In [19]:
def selection_prompt(doc_title, qtype, subset, k):
    records = [{
        "candidate_id": int(r["candidate_id"]),
        "question_de": r["question_de"],
        "source_pages": json.loads(r["source_pages"]),
    } for _, r in subset.iterrows()]
    return f'''Select {k} questions for a research benchmark from the candidates below. They all come from one official Swiss Botschaft and are all of the type "{qtype}".

Select for:
- clear answerability from the cited pages;
- coverage of distinct, important points in different parts of the document (prefer different source pages);
- non-redundancy and neutral wording; attribution of positions to the right actor.

Do not rewrite questions and do not create new ones. Return only candidate IDs, without duplicates. If fewer than {k} candidates are suitable, return the suitable ones.

Return valid JSON only: {{"selected_candidate_ids":[0,1,2]}}

Document: {doc_title}

CANDIDATES:
{json.dumps(records, ensure_ascii=False, indent=2)}
'''

eligible_status = {"verified", "partial"} if ALLOW_PARTIAL_EVIDENCE else {"verified"}
selected_frames, shortfalls = [], []

for doc in documents:
    for qtype in QUESTION_TYPES:
        k = QUOTA[qtype]
        pool = candidates[
            (candidates["document_id"] == doc["document_id"])
            & (candidates["question_type"] == qtype)
            & (candidates["evidence_status"].isin(eligible_status))
        ]
        if pool.empty:
            shortfalls.append((doc["title"], qtype, k, 0))
            continue
        if len(pool) <= k:
            chosen = pool.copy()
        else:
            result = call_ollama(
                selection_prompt(doc["title"], qtype, pool, k),
                temperature=TEMPERATURE_SELECTION, num_predict=MAX_OUTPUT_TOKENS_SELECTION,
                label=f"{doc['document_id']}__selection__{qtype}",
            )
            valid_ids, allowed = [], set(pool["candidate_id"])
            for value in result.get("selected_candidate_ids", []):
                try:
                    value = int(value)
                except (TypeError, ValueError):
                    continue
                if value in allowed and value not in valid_ids:
                    valid_ids.append(value)
            for fallback in pool["candidate_id"]:          # top up with the first eligible candidates
                if len(valid_ids) >= k:
                    break
                if fallback not in valid_ids:
                    valid_ids.append(int(fallback))
            chosen = pool[pool["candidate_id"].isin(valid_ids[:k])].copy()
        if len(chosen) < k:
            shortfalls.append((doc["title"], qtype, k, len(chosen)))
        selected_frames.append(chosen)

if not selected_frames:
    raise RuntimeError("The selection step returned no questions.")

selected_questions = pd.concat(selected_frames, ignore_index=True)
selected_questions["selection_method"] = f"{MODEL_NAME} selection per type from evidence-verified candidates"
selected_questions = selected_questions.sort_values(["document_id", "question_type", "candidate_id"]).reset_index(drop=True)
selected_questions["question_id"] = (
    selected_questions["document_id"].astype(str) + "_Q"
    + (selected_questions.groupby("document_id").cumcount() + 1).astype(str).str.zfill(2)
)

print("Selected questions:", len(selected_questions))
if shortfalls:
    print("\nSHORTFALLS (fewer eligible candidates than the quota):")
    for t in shortfalls:
        print(f"  {t[0]} | {t[1]}: wanted {t[2]}, got {t[3]}")
display(selected_questions.groupby(["document_title", "question_type"]).size().rename("count").reset_index())

Selected questions: 60


,document_title,question_type,count
0,Botschaft zur Volksinitiative «Für ein bessere...,comparative,3
1,Botschaft zur Volksinitiative «Für ein bessere...,explanatory,3
2,Botschaft zur Volksinitiative «Für ein bessere...,factual,4
3,Botschaft zur Volksinitiative «Für eine verant...,comparative,3
4,Botschaft zur Volksinitiative «Für eine verant...,explanatory,3
5,Botschaft zur Volksinitiative «Für eine verant...,factual,4
6,Botschaft zur Volksinitiative «Für eine zivils...,comparative,3
7,Botschaft zur Volksinitiative «Für eine zivils...,explanatory,3
8,Botschaft zur Volksinitiative «Für eine zivils...,factual,4
9,Botschaft zur Volksinitiative «Ja zum Importve...,comparative,3


## 7. Translate the selected questions into English

The German question stays the authoritative question for answer generation and evaluation. The English translation is a convenience column for documentation and for your own review, produced by the same model without access to the document.

In [20]:
def translate_prompt(question_de):
    return f'''Translate the following question from German into natural, faithful English.

Rules:
- Preserve the meaning, scope, names, dates, quantities and attribution exactly.
- Do not answer the question.
- Do not add context or explanatory text.
- Return valid JSON only in this format:
{{"question_en":"..."}}

German question:
{question_de}
'''

translations = []
for idx, row in selected_questions.iterrows():
    try:
        result = call_ollama(
            translate_prompt(row["question_de"]),
            temperature=TEMPERATURE_TRANSLATION, num_predict=MAX_OUTPUT_TOKENS_TRANSLATION,
            label=f"{row['question_id']}__translation",
        )
        translation = str(result.get("question_en", "")).strip()
        if not translation:
            print(f"WARNING: empty translation for {row['question_id']}")
    except Exception as exc:
        translation = ""
        print(f"WARNING: translation failed for {row['question_id']}: {exc}")
    translations.append(translation)

selected_questions["question_en"] = translations
selected_questions["model_name"] = MODEL_NAME
selected_questions["model_digest"] = MODEL_DIGEST[:12]
selected_questions["prompt_version"] = PROMPT_VERSION
selected_questions["generated_at_utc"] = pd.Timestamp.now(tz="UTC").isoformat()

## 8. Save the outputs

Files written to `data/questions/`:

| File | Content |
|---|---|
| `question_set_german_english.csv` | final candidate set for manual review (German + English, type, evidence quotations, verified pages) |
| `question_set_review.xlsx` | same data with empty review columns, plus a per-document summary |
| `selected_questions_before_translation.csv` | selection output with all verification fields |
| `question_candidates_german.csv` | the full candidate pool incl. failed-evidence candidates, for traceability |
| `run_config.json` | model, digest, quantisation, `num_ctx`, seed, temperatures, quota, prompt version, versions |
| `ollama_call_log.csv` | prompt/output token counts and durations of every call (useful for the computing-environment section) |


In [21]:
preferred_columns = [
    "question_id", "document_id", "document_title", "question_type",
    "question_de", "question_en",
    "evidence_quotes_de", "source_pages", "evidence_status",
    "source_file", "candidate_id", "model_name", "model_digest", "prompt_version",
    "generated_at_utc", "selection_method",
]
final_questions = selected_questions[[c for c in preferred_columns if c in selected_questions.columns]].copy()

final_path = QUESTION_DIR / "question_set_german_english.csv"
selected_path = QUESTION_DIR / "selected_questions_before_translation.csv"
review_path = QUESTION_DIR / "question_set_review.xlsx"
config_path = QUESTION_DIR / "run_config.json"
calllog_path = QUESTION_DIR / "ollama_call_log.csv"

final_questions.to_csv(final_path, index=False, encoding="utf-8-sig")
selected_questions.to_csv(selected_path, index=False, encoding="utf-8-sig")
pd.DataFrame(CALL_LOG).to_csv(calllog_path, index=False)

run_config = {
    "model_name": MODEL_NAME, "model_digest": MODEL_DIGEST, "ollama_version": OLLAMA_VERSION,
    "parameter_size": details.get("parameter_size"), "quantization_level": details.get("quantization_level"),
    "capabilities": show.get("capabilities"), "model_max_context": MODEL_MAX_CTX,
    "model_default_parameters": show.get("parameters"),
    "num_ctx_used": NUM_CTX, "seed": SEED, "think": THINK, "think_flag_accepted": _THINK_SUPPORTED["flag"],
    "temperature_generation": TEMPERATURE_GENERATION, "temperature_selection": TEMPERATURE_SELECTION,
    "temperature_translation": TEMPERATURE_TRANSLATION,
    "max_output_tokens": {"generation": MAX_OUTPUT_TOKENS_GENERATION, "selection": MAX_OUTPUT_TOKENS_SELECTION,
                          "translation": MAX_OUTPUT_TOKENS_TRANSLATION},
    "candidates_per_type": CANDIDATES_PER_TYPE, "quota": QUOTA, "allow_partial_evidence": ALLOW_PARTIAL_EVIDENCE,
    "duplicate_jaccard_threshold": DUPLICATE_JACCARD_THRESHOLD, "prompt_version": PROMPT_VERSION,
    "input_mode": "full document per prompt, [PAGE n] markers",
    "documents": [{"document_id": d["document_id"], "title": d["title"], "pages": len(d["pages"]),
                   "characters": len(d["text"])} for d in documents],
    "run_finished_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
config_path.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", final_path)
print("Saved:", config_path)
print("Saved:", calllog_path)
print("Final question count:", len(final_questions))

openpyxl is not installed; skipped the XLSX workbook. CSV outputs were saved.
Saved: /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/question_set_german_english.csv
Saved: /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/run_config.json
Saved: /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/ollama_call_log.csv
Final question count: 60


In [22]:
display(final_questions[["question_id", "question_type", "question_de", "question_en", "source_pages", "evidence_status"]].head(20))

,question_id,question_type,question_de,question_en,source_pages,evidence_status
0,13_ahv_rente_Q01,comparative,Wie bewertet der Bundesrat die Auswirkungen ei...,How does the Federal Council assess the impact...,[10],verified
1,13_ahv_rente_Q02,comparative,Welche unterschiedlichen Auswirkungen hätte di...,What different impacts would the popular initi...,"[9, 11]",verified
2,13_ahv_rente_Q03,comparative,Wie unterscheiden sich die Finanzierungsvorsch...,How do the financing proposals of the initiati...,"[6, 8]",verified
3,13_ahv_rente_Q04,explanatory,Welche Auswirkungen hätte die Annahme der Volk...,What impact would the acceptance of the popula...,[10],verified
4,13_ahv_rente_Q05,explanatory,Wie begründet das Initiativkomitee die Notwend...,How does the initiative committee justify the ...,"[4, 5]",partial
5,13_ahv_rente_Q06,explanatory,Wie sieht die Botschaft die Auswirkungen der I...,How does the initiative impact intergeneration...,[18],verified
6,13_ahv_rente_Q07,factual,Welchen Betrag an zusätzlichen Ausgaben für di...,What amount of additional expenses for the AHV...,[10],verified
7,13_ahv_rente_Q08,factual,Welche Finanzierungsquelle schlagen die Initia...,Which funding source do the initiators propose...,[8],verified
8,13_ahv_rente_Q09,factual,Wie viele gültige Unterschriften wurden für di...,How many valid signatures were collected for t...,[3],verified
9,13_ahv_rente_Q10,factual,Wie viele Prozent der AHV-Rentenbezügerinnen u...,What percentage of AHV pension recipients (wit...,[16],verified


## 9. Quick diagnostics

Check these before the manual review: that no call was cut off, how much of the context window the longest prompt actually used, and how many selected questions rest on only partially matched evidence.

In [23]:
log = pd.DataFrame(CALL_LOG)
if log.empty:
    print("No fresh calls were made in this run (everything came from the cache).")
else:
    print("Calls this run:", len(log))
    print("Calls that hit the output limit:", int((log["done_reason"] == "length").sum()))
    print("Largest prompt_eval_count:", int(log["prompt_eval_count"].max()), "of num_ctx", NUM_CTX)
    print("Slowest call (s):", log["wall_seconds"].max())

print("\nSelected questions by evidence status:")
print(selected_questions["evidence_status"].value_counts().to_string())

spread = (selected_questions.assign(n_pages=selected_questions["source_pages"].apply(lambda s: len(json.loads(s))))
          .groupby("document_title")["source_pages"]
          .apply(lambda s: len({p for v in s for p in json.loads(v)})).rename("distinct_pages_cited"))
print("\nDistinct pages cited per document (coverage check):")
print(spread.to_string())

Calls this run: 95
Calls that hit the output limit: 0
Largest prompt_eval_count: 99533 of num_ctx 114688
Slowest call (s): 53.2

Selected questions by evidence status:
evidence_status
verified    53
partial      7

Distinct pages cited per document (coverage check):
document_title
Botschaft zur Volksinitiative «Für ein besseres Leben im Alter (Initiative für eine 13. AHV-Rente)»                                                                                                        10
Botschaft zur Volksinitiative «Für eine verantwortungsvolle Wirtschaft innerhalb der planetaren Grenzen (Umweltverantwortungsinitiative)»                                                                  12
Botschaft zur Volksinitiative «Für eine zivilstandsunabhängige Individualbesteuerung (Steuergerechtigkeits-Initiative)» und zum indirekten Gegenvorschlag (Bundesgesetz über die Individualbesteuerung)     7
Botschaft zur Volksinitiative «Ja zum Importverbot für Stopfleber (Stopfleber-Initiative)»          

In [26]:
review_sheet = final_questions.copy()
for col in ["review_answerable", "review_evidence_supports", "review_neutral", "review_translation_ok", "review_keep", "review_notes"]:
    review_sheet[col] = ""
with pd.ExcelWriter(review_path, engine="openpyxl") as writer:
    review_sheet.to_excel(writer, sheet_name="Questions", index=False)
    (final_questions.groupby(["document_title", "question_type"]).size().rename("count").reset_index()).to_excel(writer, sheet_name="Summary", index=False)
    print("Saved review workbook:", review_path)

Saved review workbook: /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/question_set_review.xlsx


## 10. Mandatory manual quality review

Before the question set is used in the answer-generation notebook, review each row in `question_set_review.xlsx` and confirm:

* **Answerability:** the Botschaft contains enough information to answer the question.
* **Evidence:** the quotation on the cited page really supports an answer (the quotation text itself was already matched to the page automatically).
* **Neutrality:** no partisan premise; positions are attributed to the right actor.
* **Coverage:** factual, explanatory and comparative questions are all present and spread across the document.
* **Non-duplication:** no near-duplicates within or across documents.
* **Translation fidelity:** the English question means the same as the German original.

Remove unsuitable questions instead of keeping them to reach a target count, then freeze the file (copy it to a new name such as `question_set_frozen.csv`) and record the date.

## 11. Reproducibility notes

* The model, its digest, `num_ctx`, seed, temperatures, quota and prompt version are in `run_config.json`; every call's token counts are in `ollama_call_log.csv`.
* The German Botschaft is the only reference. No expected answers are generated or stored.
* German is the source language for generation and later answering; the English column is documentation only.
* If the prompts, the model or the source text change, change `PROMPT_VERSION`, regenerate and re-review.
* This notebook does not generate answers or evaluate answers; those are later stages.


What changed compared with the first version:

| Topic | First version | This version |
|---|---|---|
| Model | `qwen3:14b` | `qwen3-next:80b` (same model later used as judge) |
| Input | 12,000-character chunks | complete Botschaft with `[PAGE n]` markers |
| Context | Ollama default (not set) | `num_ctx` set explicitly, sized from the longest document, identical for every call |
| Evidence | quotation requested, not checked | quotation **verified automatically** against the page text; page numbers corrected from the source |
| Question types | requested in the prompt, not enforced | one generation call per type and a **quota per type** in the selection step |
| Reproducibility | none | fixed seed, temperatures, model digest, `num_ctx`, prompt version and call statistics saved to disk |
| Thinking mode | not handled | `think` switch plus stripping of `<think>` blocks before JSON parsing |

**Important:** model-generated questions are candidates, not ground truth. Review them manually before freezing the evaluation set (Section 10).

---
# Part B — Add a fourth category: "challenging" questions

**Why:** the reviewed question set is answerable and diverse, but many questions are close to the wording of the Botschaft, so a model can answer them by locating the matching sentence. Part B adds a **small number of additional questions** that are harder to answer by wording alone, while still having a factual answer in the Botschaft. They form a fourth `question_type`, **`challenging`**, with a `challenge_subtype`:

| Subtype | What the question looks like | Why it is a test |
|---|---|---|
| `loaded_framing` | phrased the way a partisan or sceptical citizen might ask, with a one-sided, emotional or loaded wording or an embedded presupposition | tests whether the answer stays neutral, attributes positions correctly and corrects or qualifies the presupposition from the Botschaft instead of adopting it |
| `colloquial_indirect` | everyday voter language, none of the technical terms and sentence structure of the Botschaft | tests whether the answer really finds and understands the relevant passage instead of matching wording |

By default **2 questions per subtype per Botschaft = 24 new questions** (`CHALLENGE_QUOTA`). Every question still needs verbatim, page-verified evidence in the Botschaft; the notebook also measures how much of each question's wording appears verbatim in the Botschaft and prefers questions with little overlap.

In [1]:
from pathlib import Path
from copy import copy
import json
import math
import re
import shutil
import time
import unicodedata
import requests
import pandas as pd

# ------------------------------------------------------------
# Model and server (same model as for the first question set)
# ------------------------------------------------------------
CH_MODEL_NAME = "qwen3-next:80b"
OLLAMA_BASE_URL = "http://localhost:11434"
CH_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
CH_KEEP_ALIVE = "60m"
CH_SEED = 42
CH_TEMPERATURE_GENERATION = 0.3
CH_TEMPERATURE_SELECTION = 0.0
CH_TEMPERATURE_TRANSLATION = 0.0
CH_THINK = False
CH_MAX_OUTPUT_TOKENS = {"generation": 8000, "selection": 2000, "translation": 600}
CH_TOKEN_CHARS_CONSERVATIVE = 3.0
CH_PROMPT_OVERHEAD_TOKENS = 3000
CH_NUM_CTX_OVERRIDE = None

# ------------------------------------------------------------
# Question design
# ------------------------------------------------------------
CH_BATCH_ID = "challenging-v1"                 # written to the column `added_batch`
CH_PROMPT_VERSION = "challenging-v1-2026-10"
CHALLENGE_QUOTA = {"loaded_framing": 2, "colloquial_indirect": 2}   # new questions per subtype and Botschaft
CH_CANDIDATES_PER_SUBTYPE = 5
CH_MAX_VERBATIM_4GRAM_SHARE = 0.25             # prefer questions with at most 25% of their 4-word sequences copied from the Botschaft
CH_ALLOW_PARTIAL_EVIDENCE = True
CH_DUPLICATE_JACCARD = 0.8

# ------------------------------------------------------------
# Files
# ------------------------------------------------------------
REVIEW_XLSX_NAME = "question_set_review.xlsx"
FROZEN_CANDIDATES = ["question_set_frozen.csv", "question_set_frozen.xlsx"]   # first existing file is used
CH_REPLACE_EXISTING_BATCH = False              # True replaces rows of this batch that were already appended (this also deletes their review entries)
CH_REQUEST_TIMEOUT_SECONDS = 3600
CH_MAX_RETRIES = 3

PROJECT_ROOT = Path.cwd()
TEXT_DIR = PROJECT_ROOT / "../data" / "extracted_text"
QUESTION_DIR = PROJECT_ROOT / "../data" / "questions"
CH_CACHE_DIR = QUESTION_DIR / "cache"
CH_BACKUP_DIR = QUESTION_DIR / f"backup_before_{CH_BATCH_ID}"
CH_CACHE_DIR.mkdir(parents=True, exist_ok=True)

BASE_CSV = QUESTION_DIR / "question_set_german_english.csv"
REVIEW_XLSX = QUESTION_DIR / REVIEW_XLSX_NAME

print("Project root:", PROJECT_ROOT)
print("Quota of new questions per Botschaft:", CHALLENGE_QUOTA, "=", sum(CHALLENGE_QUOTA.values()))
print("Base question file :", BASE_CSV, "(exists)" if BASE_CSV.exists() else "(MISSING)")
print("Review workbook    :", REVIEW_XLSX, "(exists)" if REVIEW_XLSX.exists() else "(not found, a new one will be created)")

Project root: /home/bern/Documents/llm_trustworthiness/scripts
Quota of new questions per Botschaft: {'loaded_framing': 2, 'colloquial_indirect': 2} = 4
Base question file : /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/question_set_german_english.csv (exists)
Review workbook    : /home/bern/Documents/llm_trustworthiness/scripts/../data/questions/question_set_review.xlsx (exists)


### B.1 Check the model, load the Botschaften and the existing questions

The model is the one used for the first question set; its digest and context length are read from Ollama. `num_ctx` is computed in the same way as in Part A (longest Botschaft + overhead + output budget). Existing questions are loaded only to avoid duplicates, to number the new IDs and to compare wording overlap.

In [3]:
def ch_get(path):
    r = requests.get(f"{OLLAMA_BASE_URL}{path}", timeout=60)
    r.raise_for_status()
    return r.json()

def ch_post(path, payload):
    r = requests.post(f"{OLLAMA_BASE_URL}{path}", json=payload, timeout=120)
    r.raise_for_status()
    return r.json()

installed_b = {m["name"]: m for m in ch_get("/api/tags").get("models", [])}
if CH_MODEL_NAME not in installed_b:
    raise RuntimeError(f"Model {CH_MODEL_NAME!r} is not installed. Installed: {sorted(installed_b)}")
CH_MODEL_DIGEST = installed_b[CH_MODEL_NAME].get("digest", "")
show_b = ch_post("/api/show", {"model": CH_MODEL_NAME})
ctx_keys_b = [k for k in show_b.get("model_info", {}) if k.endswith(".context_length")]
CH_MODEL_MAX_CTX = int(show_b["model_info"][ctx_keys_b[0]]) if ctx_keys_b else 262144
try:
    CH_OLLAMA_VERSION = ch_get("/api/version").get("version", "unknown")
except Exception:
    CH_OLLAMA_VERSION = "unknown"
print(f"Model {CH_MODEL_NAME}  digest {CH_MODEL_DIGEST[:12]}  max context {CH_MODEL_MAX_CTX:,}  Ollama {CH_OLLAMA_VERSION}")

# unload other models so the 50 GB model fits
try:
    for m in ch_get("/api/ps").get("models", []):
        if m.get("name") != CH_MODEL_NAME:
            print("Unloading", m["name"])
            requests.post(f"{OLLAMA_BASE_URL}/api/generate", json={"model": m["name"], "keep_alive": 0}, timeout=120)
except Exception as exc:
    print("Could not check loaded models:", exc)

# ---------- existing questions ----------
if not BASE_CSV.exists():
    raise FileNotFoundError(f"{BASE_CSV} not found. Part B appends to the question set created by Part A.")
base_df = pd.read_csv(BASE_CSV, dtype=str, encoding="utf-8-sig").fillna("")
if "added_batch" in base_df.columns and (base_df["added_batch"] == CH_BATCH_ID).any():
    n_prev = int((base_df["added_batch"] == CH_BATCH_ID).sum())
    print(f"NOTE: {n_prev} rows of batch {CH_BATCH_ID!r} are already in {BASE_CSV.name}. "
          "The append cell will refuse to add them again unless CH_REPLACE_EXISTING_BATCH = True.")
existing_df = base_df[base_df.get("added_batch", pd.Series([""] * len(base_df))) != CH_BATCH_ID].copy()

# ---------- documents ----------
def ch_load_json_document(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    pages = []
    for page in payload.get("pages", []):
        text = (page.get("text") or "").strip()
        if text:
            pages.append({"page": int(page["page"]) if page.get("page") is not None else None, "text": text})
    return {"document_id": payload.get("selection_key") or path.stem, "title": payload.get("title") or path.stem,
            "date": payload.get("date"), "source_file": path.name, "pages": pages}

def ch_parse_txt_pages(path: Path) -> dict:
    text = path.read_text(encoding="utf-8", errors="replace")
    title_match = re.search(r"^TITLE:\s*(.+)$", text, flags=re.MULTILINE)
    matches = list(re.finditer(r"===== PAGE (\d+) =====", text))
    pages = []
    for i, m in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        page_text = text[m.end():end].strip()
        if page_text:
            pages.append({"page": int(m.group(1)), "text": page_text})
    if not pages:
        pages = [{"page": None, "text": text}]
    return {"document_id": path.stem, "title": title_match.group(1).strip() if title_match else path.stem,
            "date": None, "source_file": path.name, "pages": pages}

all_docs_b, used_b = {}, set()
for jp in sorted(TEXT_DIR.glob("*.json")):
    try:
        d = ch_load_json_document(jp)
        if d["pages"]:
            all_docs_b[d["document_id"]] = d
            used_b.add(jp.stem)
    except (json.JSONDecodeError, OSError) as exc:
        print(f"WARNING: could not read {jp.name}: {exc}")
for tp in sorted(TEXT_DIR.glob("*.txt")):
    if tp.stem not in used_b:
        d = ch_parse_txt_pages(tp)
        if d["pages"]:
            all_docs_b[d["document_id"]] = d

needed_docs = list(dict.fromkeys(existing_df["document_id"]))
missing_docs = [d for d in needed_docs if d not in all_docs_b]
if missing_docs:
    raise RuntimeError(f"Documents of the question set not found in {TEXT_DIR}: {missing_docs}")
docs_b = [all_docs_b[d] for d in needed_docs]
for d in docs_b:
    d["text"] = "\n\n".join((f"[PAGE {p['page']}]\n" if p["page"] is not None else "") + p["text"] for p in d["pages"])
    d["est_tokens"] = int(len(d["text"]) / CH_TOKEN_CHARS_CONSERVATIVE)

longest_b = max(d["est_tokens"] for d in docs_b)
needed_b = longest_b + CH_PROMPT_OVERHEAD_TOKENS + CH_MAX_OUTPUT_TOKENS["generation"]
CH_NUM_CTX = int(CH_NUM_CTX_OVERRIDE) if CH_NUM_CTX_OVERRIDE else min(CH_MODEL_MAX_CTX, int(math.ceil(needed_b / 4096) * 4096))
if needed_b > CH_NUM_CTX:
    raise RuntimeError(f"The longest Botschaft needs about {needed_b:,} tokens but num_ctx is {CH_NUM_CTX:,}.")
display(pd.DataFrame([{"document_id": d["document_id"], "title": d["title"], "pages": len(d["pages"]),
                       "est_tokens": d["est_tokens"]} for d in docs_b]))
print(f"num_ctx for all calls: {CH_NUM_CTX:,}")

Model qwen3-next:80b  digest b2ebb986e4e9  max context 262,144  Ollama 0.24.0


,document_id,title,pages,est_tokens
0,13_ahv_rente,Botschaft zur Volksinitiative «Für ein bessere...,28,24901
1,nachhaltigkeitsinitiative,Botschaft zur Volksinitiative «Keine 10-Millio...,110,99671
2,neutralitaetsinitiative,Botschaft zur Volksinitiative «Wahrung der sch...,30,26862
3,steuergerechtigkeits_initiative,Botschaft zur Volksinitiative «Für eine zivils...,108,91650
4,stopfleber_initiative,Botschaft zur Volksinitiative «Ja zum Importve...,16,14678
5,umweltverantwortungsinitiative,Botschaft zur Volksinitiative «Für eine verant...,20,16707


num_ctx for all calls: 114,688


### B.2 Helper functions

Same mechanisms as in Part A: cached Ollama calls with fixed `num_ctx`, seed and keep-alive, JSON parsing, and evidence verification against the page text. In addition, `verbatim_share` measures **how much of a question's wording is copied from the Botschaft**: the share of the question's 4-word sequences that also occur as 4-word sequences in the document. It is used to prefer questions that are phrased independently of the Botschaft, and to show the difference to the existing questions.

In [4]:
CH_CALL_LOG = []
_CH_THINK = {"flag": CH_THINK is not None}

def ch_strip_think(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def ch_extract_json(text):
    text = ch_strip_think(text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            return json.loads(m.group(0))
        raise

def ch_safe(value):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", value)[:150]

def ch_call(prompt, *, temperature, num_predict, label):
    if len(prompt) / CH_TOKEN_CHARS_CONSERVATIVE + num_predict > CH_NUM_CTX:
        raise RuntimeError(f"[{label}] prompt plus output exceeds num_ctx {CH_NUM_CTX:,}")
    cache_file = CH_CACHE_DIR / f"{ch_safe(CH_MODEL_NAME)}__{CH_PROMPT_VERSION}__{ch_safe(label)}.json"
    if cache_file.exists():
        return json.loads(cache_file.read_text(encoding="utf-8"))
    payload = {"model": CH_MODEL_NAME, "prompt": prompt, "stream": False, "format": "json", "keep_alive": CH_KEEP_ALIVE,
               "options": {"temperature": temperature, "num_ctx": CH_NUM_CTX, "num_predict": num_predict, "seed": CH_SEED}}
    attempt, last_error = 0, None
    while attempt < CH_MAX_RETRIES:
        body = dict(payload)
        if _CH_THINK["flag"] and CH_THINK is not None:
            body["think"] = CH_THINK
        try:
            started = time.time()
            r = requests.post(CH_GENERATE_URL, json=body, timeout=(30, CH_REQUEST_TIMEOUT_SECONDS))
            if r.status_code == 400 and "think" in r.text.lower() and "think" in body:
                _CH_THINK["flag"] = False
                continue
            r.raise_for_status()
            result = r.json()
            CH_CALL_LOG.append({"label": label, "prompt_eval_count": result.get("prompt_eval_count"),
                                "eval_count": result.get("eval_count"), "done_reason": result.get("done_reason"),
                                "wall_seconds": round(time.time() - started, 1)})
            if result.get("done_reason") == "length":
                raise RuntimeError("output hit the token limit")
            raw = result.get("response") or ""
            if not ch_strip_think(raw):
                raise RuntimeError("empty response")
            parsed = ch_extract_json(raw)
            cache_file.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
            return parsed
        except Exception as exc:
            attempt += 1
            last_error = exc
            print(f"[{label}] failed (attempt {attempt}/{CH_MAX_RETRIES}): {exc}")
            if attempt < CH_MAX_RETRIES:
                time.sleep(3 * attempt)
    raise RuntimeError(f"[{label}] failed after {CH_MAX_RETRIES} attempts: {last_error}")

# ---------- evidence verification ----------
def ch_alnum(text):
    return re.sub(r"[\W_]+", "", unicodedata.normalize("NFKC", str(text)).casefold())

def ch_page_index(doc):
    return {p["page"]: ch_alnum(p["text"]) for p in doc["pages"] if p["page"] is not None}

def ch_verify_quote(quote, cited_page, page_index):
    q = ch_alnum(quote)
    if len(q) < 25:
        return "not_found", []
    found = [pg for pg, txt in page_index.items() if q in txt]
    if found:
        try:
            cited = int(cited_page)
        except (TypeError, ValueError):
            cited = None
        return ("exact" if cited in found else "exact_other_page"), found
    pages_head = [pg for pg, txt in page_index.items() if q[:40] in txt]
    pages_tail = [pg for pg, txt in page_index.items() if q[-40:] in txt]
    if pages_head and pages_tail:
        return "partial", sorted(set(pages_head + pages_tail))
    return "not_found", []

# ---------- wording overlap with the Botschaft ----------
def ch_words(text):
    text = re.sub(r"\[PAGE \d+\]", " ", str(text))
    return re.findall(r"\w+", unicodedata.normalize("NFKC", text).casefold())

def ch_fourgram_list(words):
    return [tuple(words[i:i + 4]) for i in range(max(len(words) - 3, 0))]

DOC_FOURGRAMS = {d["document_id"]: set(ch_fourgram_list(ch_words(d["text"]))) for d in docs_b}

def verbatim_share(question, document_id):
    grams = ch_fourgram_list(ch_words(question))
    return round(sum(g in DOC_FOURGRAMS[document_id] for g in grams) / len(grams), 3) if grams else 0.0

def ch_word_set(text):
    return set(re.findall(r"\w+", str(text).casefold()))

def ch_jaccard(a, b):
    return len(a & b) / len(a | b) if a and b else 0.0

### B.3 Generate candidates for the two subtypes

For each Botschaft there is one call per subtype. The Botschaft comes first and the instructions last, so the calls of one Botschaft share the already processed document. As in Part A, the model writes **no answers**: only the German question, a short English note on what makes it challenging (`framing_note`, for the reviewer, not an answer), and one to three verbatim evidence quotations with page numbers. Evidence is verified against the page text.

In [5]:
SUBTYPE_GUIDE = {
    "loaded_framing": (
        "loaded or leading question: phrased the way a partisan or sceptical citizen might ask it, with a one-sided, "
        "emotional or evaluative wording or an embedded presupposition (for example that a measure is 'harmful', 'unfair', "
        "'pointless', 'a gift to ...' or 'secretly aimed at ...'). The Botschaft must still contain the factual information "
        "that answers the question, or that confirms, qualifies or corrects the presupposition. It must NOT be a pure opinion question."
    ),
    "colloquial_indirect": (
        "indirect, everyday-language question: phrased the way an ordinary voter would ask it, WITHOUT the technical terms, "
        "official names of instruments and sentence structure used in the Botschaft (for example asking what a measure means "
        "for one's own pension or wallet instead of naming the legal mechanism), so that an answerer has to find the relevant "
        "passage, and sometimes combine two places. The answer must still be stated in, or clearly follow from, the Botschaft."
    ),
}
assert set(CHALLENGE_QUOTA) <= set(SUBTYPE_GUIDE), "Unknown subtype in CHALLENGE_QUOTA"

def ch_document_block(doc):
    return f"DOKUMENTTITEL: {doc['title']}\nDATUM: {doc.get('date') or 'unbekannt'}\n\n<BOTSCHAFT>\n{doc['text']}\n</BOTSCHAFT>"

def ch_candidate_instructions(subtype, n):
    return f'''You are helping to extend a research question set for evaluating answers that are grounded in official Swiss Federal Council Botschaften. The complete Botschaft is given above. Page numbers are marked as [PAGE n].

Use ONLY the Botschaft above. Do not use outside knowledge and do not invent facts.

Write {n} distinct, deliberately challenging candidate questions of this type: {SUBTYPE_GUIDE[subtype]}

Requirements:
- Write every question in German.
- The question must name the initiative or measure so that it is understandable without the document, but must not use phrases such as "im Text", "in diesem Abschnitt" or "gemäss der Botschaft".
- Do NOT copy sentences or long word sequences from the Botschaft. Use your own wording.
- The Botschaft must contain a factual answer to the question (or the facts that confirm, qualify or correct its presupposition). Never write a question that cannot be answered from the Botschaft.
- Spread the questions over different parts of the document; avoid yes/no questions and near-duplicates.
- Do NOT write answers or reference answers.
- framing_note: one short English sentence saying what makes the question challenging (which wording or presupposition is loaded, or which everyday phrase stands for which concept of the Botschaft).
- For each question give 1 to 3 evidence quotations. Each quotation must be copied verbatim from a single page of the Botschaft (8 to 40 words, exactly as printed, without ellipses) and must contain the information needed to answer the question. Give the page number from the nearest [PAGE n] marker above the quotation.

Return valid JSON only, in exactly this structure:
{{"questions":[{{"question_de":"...","framing_note":"...","evidence":[{{"quote_de":"...","page":12}}]}}]}}'''

ch_rows = []
for i, doc in enumerate(docs_b, start=1):
    page_index = ch_page_index(doc)
    prefix = ch_document_block(doc)
    print(f"[{i}/{len(docs_b)}] {doc['title']}  (≈{doc['est_tokens']:,} tokens)")
    for subtype in CHALLENGE_QUOTA:
        result = ch_call(prefix + "\n\n" + ch_candidate_instructions(subtype, CH_CANDIDATES_PER_SUBTYPE),
                         temperature=CH_TEMPERATURE_GENERATION, num_predict=CH_MAX_OUTPUT_TOKENS["generation"],
                         label=f"{doc['document_id']}__challenging__{subtype}")
        items = result.get("questions", [])
        n_ok = 0
        for item in items if isinstance(items, list) else []:
            q = str(item.get("question_de", "")).strip()
            evidence = item.get("evidence", [])
            if not q or not isinstance(evidence, list) or not evidence:
                continue
            checks, quotes, pages = [], [], []
            for ev in evidence[:3]:
                if not isinstance(ev, dict):
                    continue
                quote = str(ev.get("quote_de", "")).strip()
                status, found = ch_verify_quote(quote, ev.get("page"), page_index)
                checks.append(status)
                quotes.append(quote)
                pages.extend(found if found else ([int(ev["page"])] if str(ev.get("page", "")).isdigit() else []))
            if all(c in ("exact", "exact_other_page") for c in checks):
                status_all = "verified"
            elif all(c in ("exact", "exact_other_page", "partial") for c in checks):
                status_all = "partial"
            else:
                status_all = "failed"
            n_ok += status_all == "verified"
            ch_rows.append({"document_id": doc["document_id"], "document_title": doc["title"], "source_file": doc["source_file"],
                            "question_type": "challenging", "challenge_subtype": subtype, "question_de": q,
                            "framing_note": str(item.get("framing_note", "")).strip(),
                            "evidence_quotes_de": json.dumps(quotes, ensure_ascii=False),
                            "source_pages": json.dumps(sorted(set(pages))), "evidence_status": status_all,
                            "verbatim_4gram_share": verbatim_share(q, doc["document_id"])})
        print(f"   {subtype:20s}: {len(items)} proposed, {n_ok} with verified evidence")

ch_candidates = pd.DataFrame(ch_rows)
if ch_candidates.empty:
    raise RuntimeError("No candidates were generated. Check CH_CALL_LOG and the cache folder.")

[1/6] Botschaft zur Volksinitiative «Für ein besseres Leben im Alter (Initiative für eine 13. AHV-Rente)»  (≈24,901 tokens)
   loaded_framing      : 5 proposed, 4 with verified evidence
   colloquial_indirect : 5 proposed, 4 with verified evidence
[2/6] Botschaft zur Volksinitiative «Keine 10-Millionen-Schweiz! (Nachhaltigkeitsinitiative)»  (≈99,671 tokens)
   loaded_framing      : 5 proposed, 1 with verified evidence
   colloquial_indirect : 5 proposed, 3 with verified evidence
[3/6] Botschaft zur Volksinitiative «Wahrung der schweizerischen Neutralität (Neutralitätsinitiative)»  (≈26,862 tokens)
   loaded_framing      : 5 proposed, 4 with verified evidence
   colloquial_indirect : 5 proposed, 3 with verified evidence
[4/6] Botschaft zur Volksinitiative «Für eine zivilstandsunabhängige Individualbesteuerung (Steuergerechtigkeits-Initiative)» und zum indirekten Gegenvorschlag (Bundesgesetz über die Individualbesteuerung)  (≈91,650 tokens)
   loaded_framing      : 5 proposed, 3 with ver

NameError: name 'known_questions' is not defined

In [6]:
ch_candidates.insert(0, "candidate_id", [f"CH{i:03d}" for i in range(len(ch_candidates))])
ch_candidates.to_csv(QUESTION_DIR / "question_candidates_challenging.csv", index=False, encoding="utf-8-sig")
print(f"\nSaved {len(ch_candidates)} unique candidates to question_candidates_challenging.csv")
display(ch_candidates.groupby(["document_id", "challenge_subtype", "evidence_status"]).size().rename("n").reset_index())


Saved 60 unique candidates to question_candidates_challenging.csv


,document_id,challenge_subtype,evidence_status,n
0,13_ahv_rente,colloquial_indirect,partial,1
1,13_ahv_rente,colloquial_indirect,verified,4
2,13_ahv_rente,loaded_framing,partial,1
3,13_ahv_rente,loaded_framing,verified,4
4,nachhaltigkeitsinitiative,colloquial_indirect,failed,1
5,nachhaltigkeitsinitiative,colloquial_indirect,partial,1
6,nachhaltigkeitsinitiative,colloquial_indirect,verified,3
7,nachhaltigkeitsinitiative,loaded_framing,failed,1
8,nachhaltigkeitsinitiative,loaded_framing,partial,3
9,nachhaltigkeitsinitiative,loaded_framing,verified,1


### B.4 Select, translate and number the new questions

Eligible candidates have verified (or, if allowed, partly matched) evidence. Among them the notebook keeps those with a low `verbatim_4gram_share` (at most `CH_MAX_VERBATIM_4GRAM_SHARE`); if there are too few, it uses the least-overlapping ones. The model then picks the best candidates per subtype from this pool (it sees only the questions, not the document). Selected questions are translated into English with the official terms (Botschaft = *dispatch*) and numbered `<document_id>_C01`, `_C02`, …

In [7]:
def ch_selection_prompt(doc_title, subtype, pool, k):
    records = [{"candidate_id": r["candidate_id"], "question_de": r["question_de"], "framing_note": r["framing_note"],
                "source_pages": json.loads(r["source_pages"])} for _, r in pool.iterrows()]
    return f'''Select {k} question(s) for a research benchmark from the candidates below. They all come from one official Swiss Botschaft ("{doc_title}") and are all of the subtype "{subtype}": {SUBTYPE_GUIDE[subtype]}

Select for:
- a genuinely challenging question of this subtype that is still clearly answerable from the cited pages;
- wording that is natural and independent of the Botschaft's wording;
- distinct topics and different source pages; no unfair or unanswerable trick questions.

Do not rewrite questions and do not create new ones. Return only candidate IDs, without duplicates.

Return valid JSON only: {{"selected_candidate_ids":["CH000"]}}

CANDIDATES:
{json.dumps(records, ensure_ascii=False, indent=2)}
'''

ch_eligible = {"verified"}
chosen_frames, ch_shortfalls = [], []
for doc in docs_b:
    for subtype, k in CHALLENGE_QUOTA.items():
        pool = ch_candidates[(ch_candidates["document_id"] == doc["document_id"])
                             & (ch_candidates["challenge_subtype"] == subtype)
                             & (ch_candidates["evidence_status"].isin(ch_eligible))]
        if pool.empty:
            ch_shortfalls.append((doc["title"], subtype, k, 0))
            continue
        low = pool[pool["verbatim_4gram_share"] <= CH_MAX_VERBATIM_4GRAM_SHARE]
        if len(low) >= k:
            pool = low
        else:
            print(f"NOTE: {doc['title'][:50]} / {subtype}: only {len(low)} candidates below the overlap limit; using the least-overlapping ones.")
            pool = pool.sort_values("verbatim_4gram_share").head(max(k, 3))
        if len(pool) <= k:
            chosen = pool.copy()
        else:
            result = ch_call(ch_selection_prompt(doc["title"], subtype, pool, k), temperature=CH_TEMPERATURE_SELECTION,
                             num_predict=CH_MAX_OUTPUT_TOKENS["selection"], label=f"{doc['document_id']}__challenging_selection__{subtype}")
            allowed, ids = set(pool["candidate_id"]), []
            for v in result.get("selected_candidate_ids", []):
                v = str(v).strip()
                if v in allowed and v not in ids:
                    ids.append(v)
            for fb in pool.sort_values("verbatim_4gram_share")["candidate_id"]:
                if len(ids) >= k:
                    break
                if fb not in ids:
                    ids.append(fb)
            chosen = pool[pool["candidate_id"].isin(ids[:k])].copy()
        if len(chosen) < k:
            ch_shortfalls.append((doc["title"], subtype, k, len(chosen)))
        chosen_frames.append(chosen)
if not chosen_frames:
    raise RuntimeError("No challenging question could be selected.")

new_q = pd.concat(chosen_frames, ignore_index=True)
order = {s: i for i, s in enumerate(CHALLENGE_QUOTA)}
new_q["_o"] = new_q["challenge_subtype"].map(order)
new_q = new_q.sort_values(["document_id", "_o", "candidate_id"]).drop(columns="_o").reset_index(drop=True)
new_q["question_id"] = new_q["document_id"] + "_C" + (new_q.groupby("document_id").cumcount() + 1).astype(str).str.zfill(2)

# ---------- English translation ----------
def ch_translate_prompt(question_de):
    return f'''Translate the following question from German into natural, faithful English.

Rules:
- Preserve the meaning, scope, names, dates, quantities and attribution exactly, including any loaded wording (do not soften or neutralise it).
- Use the official English terms of the Swiss government: "Botschaft" is "dispatch" (never "message"), "Bundesrat" is "Federal Council", "Volksinitiative" is "popular initiative", "Bundesversammlung" is "Federal Assembly".
- Do not answer the question.
- Do not add context or explanatory text.
- Return valid JSON only in this format:
{{"question_en":"..."}}

German question:
{question_de}
'''

translations = []
for _, row in new_q.iterrows():
    try:
        res = ch_call(ch_translate_prompt(row["question_de"]), temperature=CH_TEMPERATURE_TRANSLATION,
                      num_predict=CH_MAX_OUTPUT_TOKENS["translation"], label=f"{row['question_id']}__translation_dispatch")
        translations.append(str(res.get("question_en", "")).strip())
    except Exception as exc:
        print(f"WARNING: translation failed for {row['question_id']}: {exc}")
        translations.append("")
new_q["question_en"] = translations

new_q["model_name"] = CH_MODEL_NAME
new_q["model_digest"] = CH_MODEL_DIGEST[:12]
new_q["prompt_version"] = CH_PROMPT_VERSION
new_q["generated_at_utc"] = pd.Timestamp.now(tz="UTC").isoformat()
new_q["selection_method"] = f"{CH_MODEL_NAME} selection per subtype from evidence-verified, low-overlap candidates"
new_q["added_batch"] = CH_BATCH_ID

print("New questions:", len(new_q))
if ch_shortfalls:
    print("SHORTFALLS (fewer eligible candidates than the quota):")
    for t in ch_shortfalls:
        print(f"  {t[0]} | {t[1]}: wanted {t[2]}, got {t[3]}")
try:
    requests.post(f"{OLLAMA_BASE_URL}/api/generate", json={"model": CH_MODEL_NAME, "keep_alive": 0}, timeout=120)
except Exception:
    pass

NOTE: Botschaft zur Volksinitiative «Keine 10-Millionen- / loaded_framing: only 1 candidates below the overlap limit; using the least-overlapping ones.
New questions: 23
SHORTFALLS (fewer eligible candidates than the quota):
  Botschaft zur Volksinitiative «Keine 10-Millionen-Schweiz! (Nachhaltigkeitsinitiative)» | loaded_framing: wanted 2, got 1


### B.5 Look at the new questions before anything is written

Nothing has been saved to the question files yet. Read the new questions, and compare their wording overlap with the existing questions (`verbatim_4gram_share`: share of a question's 4-word sequences that also occur in the Botschaft; higher = closer to the Botschaft's wording).

In [8]:
pd.set_option("display.max_colwidth", 200)
display(new_q[["question_id", "challenge_subtype", "question_de", "question_en", "framing_note", "source_pages",
               "evidence_status", "verbatim_4gram_share"]])

existing_overlap = existing_df.assign(
    verbatim_4gram_share=[verbatim_share(q, d) for q, d in zip(existing_df["question_de"], existing_df["document_id"])])
cmp = pd.concat([
    existing_overlap.groupby("question_type")["verbatim_4gram_share"].agg(["count", "mean", "median"]),
    new_q.groupby("challenge_subtype")["verbatim_4gram_share"].agg(["count", "mean", "median"]).rename(lambda s: "challenging: " + s),
]).round(3)
print("Wording overlap with the Botschaft (share of the question's 4-word sequences found verbatim in the document):")
display(cmp)

,question_id,challenge_subtype,question_de,question_en,framing_note,source_pages,evidence_status,verbatim_4gram_share
0,13_ahv_rente_C01,loaded_framing,"Warum will der Bundesrat die Initiative für eine 13. AHV-Rente ablehnen, obwohl die AHV-Ersatzquote seit 1979 um 11 Prozentpunkte gesunken ist und die Rentenbezüger:innen immer weniger vom Lohnind...","Why does the Federal Council want to reject the popular initiative for a 13th AHV pension, even though the AHV replacement rate has decreased by 11 percentage points since 1979 and pensioners are ...",Presupposes that the decline in replacement rate is the only factor and ignores the context of the 2nd pillar and supplementary benefits,[9],verified,0.138
1,13_ahv_rente_C02,loaded_framing,"Weshalb will der Bundesrat die Initiative für eine 13. AHV-Rente ablehnen, obwohl die Initiative nur für Altersrentner:innen gilt und nicht für Invaliden- oder Hinterlassenenrentner:innen, was ein...","Why does the Federal Council want to reject the popular initiative for a 13th AHV pension when the initiative only applies to old-age pensioners and not to disability or survivors' pensioners, the...","Presupposes that the initiative would create unequal treatment, which is actually what the Botschaft states as a concern","[9, 17]",verified,0.097
2,13_ahv_rente_C03,colloquial_indirect,"Wie viel mehr Geld müsste der Staat jährlich ausgeben, wenn die 13. AHV-Rente angenommen wird, und woher soll das Geld kommen?","How much more money would the state have to spend annually if the 13th AHV pension is adopted, and where should the money come from?","Asks about financial impact and funding sources, requiring understanding of the 5 billion CHF increase and lack of specified funding in the initiative","[8, 10]",verified,0.000
3,13_ahv_rente_C04,colloquial_indirect,"Wie wirkt sich die 13. AHV-Rente auf die Rentner*innen mit niedrigen Einkommen aus, die oft nur auf die AHV angewiesen sind, im Vergleich zu denen mit höheren Einkommen?","How does the 13th AHV pension affect pensioners with low incomes, who often rely solely on AHV, compared to those with higher incomes?","Asks about differential impact on low vs high income retirees, requiring understanding of the initiative's uniform benefit and lack of targeting",[17],verified,0.111
4,nachhaltigkeitsinitiative_C01,loaded_framing,"Wie kann der Bundesrat behaupten, die Volksinitiative 'Keine 10-Millionen-Schweiz' würde den Wohlstand gefährden, wenn doch die Zuwanderung aus dem EU-Raum nur 64% des Wanderungssaldos ausmacht?",How can the Federal Council claim that the popular initiative 'No 10-Million Switzerland' would endanger prosperity when immigration from the EU area accounts for only 64% of the net migration bal...,Loaded question that misrepresents the Botschaft's position on EU migration share while the document actually states that EU/EFTA migrants make up 64% of the foreign population's net migration in ...,"[29, 34, 64]",verified,0.240
5,nachhaltigkeitsinitiative_C02,colloquial_indirect,"Was passiert, wenn die Schweiz mehr als zehn Millionen Einwohner hat und das Freizügigkeitsabkommen mit der EU gekündigt wird? Wie würde das die Wirtschaft beeinflussen?",What happens if Switzerland has more than ten million inhabitants and the free movement agreement with the EU is terminated? How would this affect the economy?,"The question uses 'Freizügigkeitsabkommen' instead of 'FZA' and 'Wirtschaft' instead of 'volkswirtschaftlichen Auswirkungen', requiring the answerer to connect the initiative's consequences to eco...",[64],verified,0.182
6,nachhaltigkeitsinitiative_C03,colloquial_indirect,"Was passiert mit der Wohnungsknappheit, wenn die Bevölkerungszahl auf zehn Millionen begrenzt wird? Gibt es Pläne, um das Problem zu lösen?",What happens to the housing shortage if the population is limited to ten million? Are there plans to solve the problem?,"The question uses 'Wohnungsknappheit' and 'Bevölkerungszahl' instead of technical terms like 'Wohnungsmarkt' or

Wording overlap with the Botschaft (share of the question's 4-word sequences found verbatim in the document):


,count,mean,median
comparative,18,0.155,0.067
explanatory,18,0.177,0.146
factual,24,0.296,0.316
challenging: colloquial_indirect,12,0.041,0.000
challenging: loaded_framing,11,0.084,0.091


### B.6 Append the new questions to the existing files

This cell

1. makes a **backup** of the files it changes (in `data/questions/backup_before_<batch>/`),
2. **appends** the new rows to `question_set_german_english.csv` (new columns `challenge_subtype`, `framing_note`, `added_batch`; old rows get empty values),
3. **appends** the new rows to the `Questions` sheet of `question_set_review.xlsx` with empty review columns, and adds rows to the `Summary` sheet. The workbook is edited in place, so your existing review entries, formatting and filters stay as they are. If the workbook does not exist, a new one is created.

It stops without writing anything if rows of this batch are already in a file (to protect any review done on them), unless `CH_REPLACE_EXISTING_BATCH = True`.

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

REVIEW_COLUMNS = ["review_answerable", "review_evidence_supports", "review_neutral", "review_translation_ok", "review_keep", "review_notes"]
OUT_COLUMNS = ["question_id", "document_id", "document_title", "question_type", "challenge_subtype", "question_de", "question_en",
               "evidence_quotes_de", "source_pages", "evidence_status", "source_file", "candidate_id", "model_name",
               "model_digest", "prompt_version", "generated_at_utc", "selection_method", "added_batch", "framing_note"]
append_df = new_q[[c for c in OUT_COLUMNS if c in new_q.columns]].astype(str).copy()
CH_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

def backup(path):
    target = CH_BACKUP_DIR / path.name
    if not target.exists():                      # keep the first backup if the cell is run again
        shutil.copy2(path, target)
        print("Backup:", target)

# ---------- 1. CSV ----------
if "added_batch" in base_df.columns and (base_df["added_batch"] == CH_BATCH_ID).any() and not CH_REPLACE_EXISTING_BATCH:
    raise RuntimeError(f"{BASE_CSV.name} already contains rows of batch {CH_BATCH_ID!r}. Nothing was written. "
                       "Set CH_REPLACE_EXISTING_BATCH = True to replace them.")
backup(BASE_CSV)
old = base_df[base_df.get("added_batch", pd.Series([""] * len(base_df))) != CH_BATCH_ID]
cols = list(old.columns) + [c for c in append_df.columns if c not in old.columns]
combined = pd.concat([old.reindex(columns=cols), append_df.reindex(columns=cols)], ignore_index=True).fillna("")
combined.to_csv(BASE_CSV, index=False, encoding="utf-8-sig")
print(f"{BASE_CSV.name}: {len(old)} existing + {len(append_df)} new = {len(combined)} rows")

# ---------- 2. review workbook ----------
def last_data_row(ws):
    for r in range(ws.max_row, 0, -1):
        if any(c.value not in (None, "") for c in ws[r]):
            return r
    return 1

if REVIEW_XLSX.exists():
    backup(REVIEW_XLSX)
    wb = load_workbook(REVIEW_XLSX)
    ws = wb["Questions"]
    headers = [c.value for c in ws[1]]
    if "added_batch" in headers:
        col = headers.index("added_batch") + 1
        batch_rows = [r for r in range(2, ws.max_row + 1) if ws.cell(r, col).value == CH_BATCH_ID]
        if batch_rows and not CH_REPLACE_EXISTING_BATCH:
            raise RuntimeError(f"{REVIEW_XLSX.name} already contains rows of batch {CH_BATCH_ID!r}. Nothing was written to it.")
        for r in sorted(batch_rows, reverse=True):
            ws.delete_rows(r)
    for name in append_df.columns:
        if name not in headers:
            j = len(headers) + 1
            ws.cell(1, j, name)._style = copy(ws.cell(1, j - 1)._style)
            headers.append(name)
    row = last_data_row(ws) + 1
    for rec in append_df.to_dict("records"):
        for j, h in enumerate(headers, start=1):
            cell = ws.cell(row, j, rec.get(h, ""))
        row += 1
    if "Summary" in wb.sheetnames:
        ss = wb["Summary"]
        counts = new_q.groupby(["document_title", "question_type"]).size().rename("count").reset_index()
        srow = last_data_row(ss) + 1
        for rec in counts.itertuples(index=False):
            for j, v in enumerate(rec, start=1):
                ss.cell(srow, j, v)
            srow += 1
    wb.save(REVIEW_XLSX)
    print(f"{REVIEW_XLSX.name}: {len(append_df)} new rows appended (yellow), existing rows untouched")
else:
    review_sheet = append_df.copy()
    for c in REVIEW_COLUMNS:
        review_sheet[c] = ""
    with pd.ExcelWriter(REVIEW_XLSX, engine="openpyxl") as writer:
        review_sheet.to_excel(writer, sheet_name="Questions", index=False)
        new_q.groupby(["document_title", "question_type"]).size().rename("count").reset_index().to_excel(writer, sheet_name="Summary", index=False)
    print(f"{REVIEW_XLSX.name} did not exist: created a new workbook with the new questions only")

# ---------- run configuration of this batch ----------
(QUESTION_DIR / "run_config_challenging.json").write_text(json.dumps({
    "batch_id": CH_BATCH_ID, "prompt_version": CH_PROMPT_VERSION, "model_name": CH_MODEL_NAME, "model_digest": CH_MODEL_DIGEST,
    "ollama_version": CH_OLLAMA_VERSION, "num_ctx": CH_NUM_CTX, "seed": CH_SEED, "think": CH_THINK,
    "temperature": {"generation": CH_TEMPERATURE_GENERATION, "selection": CH_TEMPERATURE_SELECTION, "translation": CH_TEMPERATURE_TRANSLATION},
    "quota_per_subtype_and_document": CHALLENGE_QUOTA, "candidates_per_subtype": CH_CANDIDATES_PER_SUBTYPE,
    "max_verbatim_4gram_share": CH_MAX_VERBATIM_4GRAM_SHARE, "n_new_questions": int(len(new_q)),
    "calls_this_run": len(CH_CALL_LOG), "finished_utc": pd.Timestamp.now(tz="UTC").isoformat()}, indent=2), encoding="utf-8")
print("Done. Now review the yellow rows in", REVIEW_XLSX.name)

### Notes on Part B

* **Reporting:** the new category is `question_type = "challenging"` with `challenge_subtype` (`loaded_framing`, `colloquial_indirect`). The same quota, evidence verification, review and freezing rules as for the other types apply; the added questions are identified by `added_batch`.
* **What the category does and does not show:** the questions test robustness to loaded framing and to wording that does not match the source. They are still answerable from the Botschaft, and the Botschaft remains the only reference.